In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gc 
from sklearn.metrics import f1_score
import json

import sys
import os

sys.path.append(os.path.abspath(os.path.join('..')))

from src.load_data_gait import *
from src.plot_data_gait import *
from src.train_LSTMGait import train_model, predict_trial

In [2]:
base_path = '../datasets/GAIT_signals/dataset/data'
X, y, subjects = load_dataset_gait(base_path, process="free_raw", sensors=['LB'], window_size=200, stride=50)
X = X.astype(np.float32)
y = y.astype(np.int64)

Trials descartados: 10 ['HS_2_1', 'KOA_17_3', 'KOA_17_4', 'PD_6_4', 'PD_6_5', 'RIL_24_6', 'RIL_24_7', 'RIL_24_9', 'RIL_31_12', 'RIL_42_2'] | sujetos con datos: 260


# Training

In [3]:
results = []
results_path = "../checkpoints/model_refination.json"

if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)

for lr in [0.001, 0.0003]:
    for dropout in [0.3, 0.5]:
        for conv_kernel in [5, 7]:
            for batch_size in [128, 256]:
                for model_deep in [(64,128,128)]:

                    print(f"Processing with lr={lr}, dropout={dropout}, conv_kernel={conv_kernel}, batch_size={batch_size}, model_deep={model_deep}")

                    ckpt_path = f"../checkpoints/lr{lr}_dropout{dropout}_kernel{conv_kernel}_batch{batch_size}_deep{model_deep}_hidden{64}_layers{1}"
                    trained_model, best_val_f1, test_f1 = train_model(X, y, subjects, model_name="cnnbilstm", norm="window", epochs=200, lr=lr, patience=20, dropout=dropout, cnn_channels=model_deep, kernel_size=conv_kernel, batch_size=batch_size, hidden_size=64, num_layers=1, out_dir=ckpt_path)


                    results.append({
                        "lr": lr,
                        "dropout": dropout,
                        "conv_kernel": conv_kernel,
                        "batch_size": batch_size,
                        "model_deep": model_deep,
                        "hidden_size": 64,
                        "num_layers": 1,
                        "val_f1": round(float(best_val_f1), 6),
                        "test_f1": round(float(test_f1), 6),
                        "checkpoint": ckpt_path,
                    })
                    print("")
                    print("================================")
                    print("")

                    with open(results_path, 'w') as f:
                        json.dump(results, f, indent=2)

                    del trained_model
                    gc.collect()

print(f"\nDone. {len(results)} combinations trained.")

Processing with lr=0.001, dropout=0.3, conv_kernel=5, batch_size=128, model_deep=(64, 128, 128)
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 58845  |  test: 16995
               →  train: 166 subjects (45819 windows)  |  val: 42 subjects (13026 windows)


Early stopping in epoch 57. Best F1: 0.9188

Val F1: 0.9188  |  Test F1: 0.9025
Checkpoint saved → ../checkpoints/lr0.001_dropout0.3_kernel5_batch128_deep(64, 128, 128)_hidden64_layers1/model.pt


Processing with lr=0.001, dropout=0.3, conv_kernel=5, batch_size=256, model_deep=(64, 128, 128)
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 58845  |  test: 16995
               →  train: 166 subjects (45819 windows)  |  val: 42 subjects (13026 windows)


Early stopping in epoch 87. Best F1: 0.9184

Val F1: 0.9184  |  Test F1: 0.8998
Checkpoint saved → ../checkpoints/lr0.001_dropout0.3_kernel5_batch256_deep(64, 128, 128)_hidden64_layers1/model.pt


Processing with lr=0.001, dropout=0.3, conv_kernel=7, batch_size=128, model_deep=(64, 128, 128)
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 58845  |  test: 16995
               →  train: 166 subjects (45819 windows)  |  val: 42 subjects (13026 windows)


Early stopping in epoch 87. Best F1: 0.9189

Val F1: 0.9189  |  Test F1: 0.8970
Checkpoint saved → ../checkpoints/lr0.001_dropout0.3_kernel7_batch128_deep(64, 128, 128)_hidden64_layers1/model.pt


Processing with lr=0.001, dropout=0.3, conv_kernel=7, batch_size=256, model_deep=(64, 128, 128)
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 58845  |  test: 16995
               →  train: 166 subjects (45819 windows)  |  val: 42 subjects (13026 windows)


Early stopping in epoch 63. Best F1: 0.9186

Val F1: 0.9186  |  Test F1: 0.9001
Checkpoint saved → ../checkpoints/lr0.001_dropout0.3_kernel7_batch256_deep(64, 128, 128)_hidden64_layers1/model.pt


Processing with lr=0.001, dropout=0.5, conv_kernel=5, batch_size=128, model_deep=(64, 128, 128)
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 58845  |  test: 16995
               →  train: 166 subjects (45819 windows)  |  val: 42 subjects (13026 windows)


...

Early stopping in epoch 110. Best F1: 0.9179

Val F1: 0.9179  |  Test F1: 0.8987
Checkpoint saved → ../checkpoints/lr0.0003_dropout0.3_kernel7_batch256_deep(64, 128, 128)_hidden64_layers1/model.pt


Processing with lr=0.0003, dropout=0.5, conv_kernel=5, batch_size=128, model_deep=(64, 128, 128)
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 58845  |  test: 16995
               →  train: 166 subjects (45819 windows)  |  val: 42 subjects (13026 windows)


Early stopping in epoch 84. Best F1: 0.9181

Val F1: 0.9181  |  Test F1: 0.8959
Checkpoint saved → ../checkpoints/lr0.0003_dropout0.5_kernel5_batch128_deep(64, 128, 128)_hidden64_layers1/model.pt


Processing with lr=0.0003, dropout=0.5, conv_kernel=5, batch_size=256, model_deep=(64, 128, 128)
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 58845  |  test: 16995
               →  train: 166 subjects (45819 windows)  |  val: 42 subjects (13026 windows)


Early stopping in epoch 80. Best F1: 0.9174

Val F1: 0.9174  |  Test F1: 0.8959
Checkpoint saved → ../checkpoints/lr0.0003_dropout0.5_kernel5_batch256_deep(64, 128, 128)_hidden64_layers1/model.pt


Processing with lr=0.0003, dropout=0.5, conv_kernel=7, batch_size=128, model_deep=(64, 128, 128)
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 58845  |  test: 16995
               →  train: 166 subjects (45819 windows)  |  val: 42 subjects (13026 windows)


Early stopping in epoch 80. Best F1: 0.9194

Val F1: 0.9194  |  Test F1: 0.8993
Checkpoint saved → ../checkpoints/lr0.0003_dropout0.5_kernel7_batch128_deep(64, 128, 128)_hidden64_layers1/model.pt


Processing with lr=0.0003, dropout=0.5, conv_kernel=7, batch_size=256, model_deep=(64, 128, 128)
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 58845  |  test: 16995
               →  train: 166 subjects (45819 windows)  |  val: 42 subjects (13026 windows)


Early stopping in epoch 104. Best F1: 0.9196

Val F1: 0.9196  |  Test F1: 0.9023
Checkpoint saved → ../checkpoints/lr0.0003_dropout0.5_kernel7_batch256_deep(64, 128, 128)_hidden64_layers1/model.pt



Done. 65 combinations trained.


# Results comparison

In [4]:
import json

with open("../checkpoints/model_refination.json", "r") as f:
    results = json.load(f)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("val_f1", ascending=False).reset_index(drop=True)
results_df

,lr,dropout,conv_kernel,batch_size,model_deep,hidden_size,num_layers,val_f1,test_f1,checkpoint
0,0.0010,0.5,7,128,"[64, 128, 128]",64,1,0.919676,0.901261,../checkpoints/lr0.001_dropout0.5_kernel7_batc...
1,0.0010,0.5,7,128,"[64, 128, 128]",64,1,0.919676,0.901261,../checkpoints/lr0.001_dropout0.5_kernel7_batc...
2,0.0003,0.5,7,256,"[64, 128, 128]",64,1,0.919600,0.902293,../checkpoints/lr0.0003_dropout0.5_kernel7_bat...
3,0.0010,0.5,7,256,"[64, 128, 128]",64,1,0.919518,0.896521,../checkpoints/lr0.001_dropout0.5_kernel7_batc...
4,0.0010,0.5,7,256,"[64, 128, 128]",64,1,0.919518,0.896521,../checkpoints/lr0.001_dropout0.5_kernel7_batc...
...,...,...,...,...,...,...,...,...,...,...
60,0.0010,0.5,7,256,"[64, 128]",64,1,0.858778,0.868180,../checkpoints/lr0.001_dropout0.5_kernel7_batc...
61,0.0010,0.3,7,256,"[64, 128, 128]",64,1,0.858770,0.868639,../checkpoints/lr0.001_dropout0.3_kernel7_batc...
62,0.0010,0.5,7,256,"[64, 128]",64,2,0.858207,0.870108,../checkpoints/lr0.001_dropout0.5_kernel7_batc...
63,0.0010,0.5,7,256,"[64, 128]",256,1,0.857539,0.867513,../checkpoints/lr0.001_dropout0.5_kernel7_batc...


In [5]:
best_row = results_df.iloc[0]
print("Best configuration (selected by val_f1):")
print(best_row)
print(f"\n--> Reported test_f1 for this configuration: {best_row['test_f1']:.4f}")

Best configuration (selected by val_f1):
lr                                                         0.001
dropout                                                      0.5
conv_kernel                                                    7
batch_size                                                   128
model_deep                                        [64, 128, 128]
hidden_size                                                   64
num_layers                                                     1
val_f1                                                  0.919676
test_f1                                                 0.901261
checkpoint     ../checkpoints/lr0.001_dropout0.5_kernel7_batc...
Name: 0, dtype: object

--> Reported test_f1 for this configuration: 0.9013


# Training

In [3]:
results = []
results_path = "../checkpoints/model_refination.json"

if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)

top4_confs = [
    (0.001, 0.5, 7, 128, (64, 128, 128)),
    (0.001, 0.5, 7, 256, (64, 128, 128)),
]

for conf in top4_confs:
    for hidden_size in [128, 256]:
        for num_layers in [1, 2, 3]:

            print(f"Processing with conf={conf}, hidden={hidden_size}, num_layers={num_layers}")

            ckpt_path = f"../checkpoints/lr{conf[0]}_dropout{conf[1]}_kernel{conf[2]}_batch{conf[3]}_deep{conf[4]}_hidden{hidden_size}_layers{num_layers}"
            trained_model, best_val_f1, test_f1 = train_model(X, y, subjects, model_name="cnnbilstm", norm="window", epochs=200, lr=conf[0], patience=20, dropout=conf[1], cnn_channels=conf[4], kernel_size=conf[2], batch_size=conf[3], hidden_size=hidden_size, num_layers=num_layers, out_dir=ckpt_path)


            results.append({
                "lr": conf[0],
                "dropout": conf[1],
                "conv_kernel": conf[2],
                "batch_size": conf[3],
                "model_deep": conf[4],
                "hidden_size": hidden_size,
                "num_layers": num_layers,
                "val_f1": round(float(best_val_f1), 6),
                "test_f1": round(float(test_f1), 6),
                "checkpoint": ckpt_path,
            })
            print("")
            print("================================")
            print("")

            with open(results_path, 'w') as f:
                json.dump(results, f, indent=2)

            del trained_model
            gc.collect()    

print(f"\nDone. {len(results)} combinations trained.")

Processing with conf=(0.001, 0.5, 7, 128, (64, 128, 128)), hidden=128, num_layers=1
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 58845  |  test: 16995
               →  train: 166 subjects (45819 windows)  |  val: 42 subjects (13026 windows)


Early stopping in epoch 59. Best F1: 0.9198

Val F1: 0.9198  |  Test F1: 0.9003
Checkpoint saved → ../checkpoints/lr0.001_dropout0.5_kernel7_batch128_deep(64, 128, 128)_hidden128_layers1/model.pt


Processing with conf=(0.001, 0.5, 7, 128, (64, 128, 128)), hidden=128, num_layers=2
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 58845  |  test: 16995
               →  train: 166 subjects (45819 windows)  |  val: 42 subjects (13026 windows)


Early stopping in epoch 56. Best F1: 0.9215

Val F1: 0.9215  |  Test F1: 0.9036
Checkpoint saved → ../checkpoints/lr0.001_dropout0.5_kernel7_batch128_deep(64, 128, 128)_hidden128_layers2/model.pt


Processing with conf=(0.001, 0.5, 7, 128, (64, 128, 128)), hidden=128, num_layers=3
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 58845  |  test: 16995
               →  train: 166 subjects (45819 windows)  |  val: 42 subjects (13026 windows)


Early stopping in epoch 78. Best F1: 0.9224

Val F1: 0.9224  |  Test F1: 0.9021
Checkpoint saved → ../checkpoints/lr0.001_dropout0.5_kernel7_batch128_deep(64, 128, 128)_hidden128_layers3/model.pt


Processing with conf=(0.001, 0.5, 7, 128, (64, 128, 128)), hidden=256, num_layers=1
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 58845  |  test: 16995
               →  train: 166 subjects (45819 windows)  |  val: 42 subjects (13026 windows)


Early stopping in epoch 62. Best F1: 0.9192

Val F1: 0.9192  |  Test F1: 0.8979
Checkpoint saved → ../checkpoints/lr0.001_dropout0.5_kernel7_batch128_deep(64, 128, 128)_hidden256_layers1/model.pt


Processing with conf=(0.001, 0.5, 7, 128, (64, 128, 128)), hidden=256, num_layers=2
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 58845  |  test: 16995
               →  train: 166 subjects (45819 windows)  |  val: 42 subjects (13026 windows)


...

Early stopping in epoch 73. Best F1: 0.9209

Val F1: 0.9209  |  Test F1: 0.9017
Checkpoint saved → ../checkpoints/lr0.001_dropout0.5_kernel7_batch256_deep(64, 128, 128)_hidden128_layers2/model.pt


Processing with conf=(0.001, 0.5, 7, 256, (64, 128, 128)), hidden=128, num_layers=3
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 58845  |  test: 16995
               →  train: 166 subjects (45819 windows)  |  val: 42 subjects (13026 windows)


Early stopping in epoch 66. Best F1: 0.9225

Val F1: 0.9225  |  Test F1: 0.9028
Checkpoint saved → ../checkpoints/lr0.001_dropout0.5_kernel7_batch256_deep(64, 128, 128)_hidden128_layers3/model.pt


Processing with conf=(0.001, 0.5, 7, 256, (64, 128, 128)), hidden=256, num_layers=1
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 58845  |  test: 16995
               →  train: 166 subjects (45819 windows)  |  val: 42 subjects (13026 windows)


Early stopping in epoch 68. Best F1: 0.9209

Val F1: 0.9209  |  Test F1: 0.9044
Checkpoint saved → ../checkpoints/lr0.001_dropout0.5_kernel7_batch256_deep(64, 128, 128)_hidden256_layers1/model.pt


Processing with conf=(0.001, 0.5, 7, 256, (64, 128, 128)), hidden=256, num_layers=2
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 58845  |  test: 16995
               →  train: 166 subjects (45819 windows)  |  val: 42 subjects (13026 windows)


Early stopping in epoch 56. Best F1: 0.9204

Val F1: 0.9204  |  Test F1: 0.8953
Checkpoint saved → ../checkpoints/lr0.001_dropout0.5_kernel7_batch256_deep(64, 128, 128)_hidden256_layers2/model.pt


Processing with conf=(0.001, 0.5, 7, 256, (64, 128, 128)), hidden=256, num_layers=3
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 58845  |  test: 16995
               →  train: 166 subjects (45819 windows)  |  val: 42 subjects (13026 windows)


Early stopping in epoch 84. Best F1: 0.9221

Val F1: 0.9221  |  Test F1: 0.9014
Checkpoint saved → ../checkpoints/lr0.001_dropout0.5_kernel7_batch256_deep(64, 128, 128)_hidden256_layers3/model.pt



Done. 89 combinations trained.


In [4]:
import json

with open("../checkpoints/model_refination.json", "r") as f:
    results = json.load(f)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("val_f1", ascending=False).reset_index(drop=True)
results_df

,lr,dropout,conv_kernel,batch_size,model_deep,hidden_size,num_layers,val_f1,test_f1,checkpoint
0,0.001,0.5,7,256,"[64, 128, 128]",128,3,0.922494,0.902829,../checkpoints/lr0.001_dropout0.5_kernel7_batc...
1,0.001,0.5,7,256,"[64, 128, 128]",128,3,0.922494,0.902829,../checkpoints/lr0.001_dropout0.5_kernel7_batc...
2,0.001,0.5,7,128,"[64, 128, 128]",128,3,0.922413,0.902078,../checkpoints/lr0.001_dropout0.5_kernel7_batc...
3,0.001,0.5,7,128,"[64, 128, 128]",128,3,0.922413,0.902078,../checkpoints/lr0.001_dropout0.5_kernel7_batc...
4,0.001,0.5,7,128,"[64, 128, 128]",256,3,0.922222,0.899796,../checkpoints/lr0.001_dropout0.5_kernel7_batc...
...,...,...,...,...,...,...,...,...,...,...
84,0.001,0.5,7,256,"[64, 128]",64,1,0.858778,0.868180,../checkpoints/lr0.001_dropout0.5_kernel7_batc...
85,0.001,0.3,7,256,"[64, 128, 128]",64,1,0.858770,0.868639,../checkpoints/lr0.001_dropout0.3_kernel7_batc...
86,0.001,0.5,7,256,"[64, 128]",64,2,0.858207,0.870108,../checkpoints/lr0.001_dropout0.5_kernel7_batc...
87,0.001,0.5,7,256,"[64, 128]",256,1,0.857539,0.867513,../checkpoints/lr0.001_dropout0.5_kernel7_batc...


In [5]:
best_row = results_df.iloc[0]
print("Best configuration (selected by val_f1):")
print(best_row)
print(f"\n--> Reported test_f1 for this configuration: {best_row['test_f1']:.4f}")

Best configuration (selected by val_f1):
lr                                                         0.001
dropout                                                      0.5
conv_kernel                                                    7
batch_size                                                   256
model_deep                                        [64, 128, 128]
hidden_size                                                  128
num_layers                                                     3
val_f1                                                  0.922494
test_f1                                                 0.902829
checkpoint     ../checkpoints/lr0.001_dropout0.5_kernel7_batc...
Name: 0, dtype: object

--> Reported test_f1 for this configuration: 0.9028
